In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import asyncio
import nest_asyncio
from pathlib import Path
import sys
from google.adk.apps import App
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types

PROJECT_ROOT = Path.cwd().parent
print("PROJECT ROOT ==>", PROJECT_ROOT)

sys.path.append(str(PROJECT_ROOT))
from src.mesh_fun import (
    load_segmentation
)
from src.landmark_eval import( 
    compute_all_statistics,
    quality_profile_for_scan,
select_examples_by_quantile, 
compute_profiles_for_scans)
from src.pred_gt_viewer_viz import load_pred_landmarks,visualize_pred_and_gt
from src.var_constants import CATEGORIES,TOOTH_GROUP_PALETTE,LANDMARK_PALETTE,TOOTH_TO_GROUP


PROJECT ROOT ==> c:\Users\aless\Tesi_Msc\multi_agent_evaluation_on_intraoral_3D_scans
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
CATEGORIES ==> ['Mesial', 'Distal', 'Cusp', 'InnerPoint', 'OuterPoint', 'FacialPoint']


In [5]:
#PROJECT_ROOT = Path(__file__).resolve().parent
DATA_ROOT = PROJECT_ROOT / "dataset"
SCANS = DATA_ROOT / "toothinstancenet_input"
GT_ROOT = SCANS
PRED_CSV = DATA_ROOT / "model_predictions" / "ynlab" / "predictions.csv"
SCREENSHOT_DIR = DATA_ROOT / "screenshot_scans" / "raw"

In [ ]:
def build_input_dataset(results, exclude_scans, SCANS, GT_ROOT, PRED_CSV,SCREENSHOT_ROOT):
    """
    Costruisce dataset di input escludendo le scans specificate.
    Le predizioni vengono caricate direttamente dal CSV.
    """
    input_scans = [s for s in results["scans"] if s not in exclude_scans]
    dataset = []
    for scan in input_scans:
        profile = quality_profile_for_scan(results, CATEGORIES, scan)
        #carica predizioni dal CSV
        coords_pred, classes_pred = load_pred_landmarks(PRED_CSV, scan)
        dataset.append({
            "scan": scan,
            "profile": profile,
            #paths
            "mesh": str(SCANS / f"{scan}.obj"),
            "gt": str(GT_ROOT / f"{scan}__kpt.json"),
            "seg": str(SCANS / f"{scan}_seg.json"),
            "image_pred": str(SCREENSHOT_ROOT / f"{scan}/predicted.png"),
            #predizioni dal CSV
            "pred_coords": coords_pred,
            "pred_classes": classes_pred,
        })

    return dataset


### costruisci dataset per sistema agentico

In [13]:
#calcola sia statistiche globali che per categoria, e restituisce anche la tabella dei quantili
stats = compute_all_statistics(GT_ROOT, PRED_CSV, CATEGORIES)
stats_over_scan= stats["results"]
quantile_mAP_over_scan = stats["quantile_mAP_over_scan"]

#### crea screenshot per ogni scansione, con predizioni e GT, da salvare in SCREENSHOT_DIR

In [ ]:
for scan_name in stats_over_scan["scans"]:
    #Visualizzazione predizioni + GT + screenshot
    screenshot_mesh_path =SCREENSHOT_DIR / scan_name
    mesh_path = SCANS / f"{scan_name}.obj"
    gt_json = GT_ROOT / f"{scan_name}__kpt.json"
    seg_path  =SCANS / f"{scan_name}_seg.json"
    tooth_seg_labels = load_segmentation(seg_path)
    visualize_pred_and_gt(mesh_path, PRED_CSV, gt_json, scan_name, screenshot_mesh_path, LANDMARK_PALETTE, tooth_seg_labels, TOOTH_TO_GROUP, TOOTH_GROUP_PALETTE)


#### selezione subset scan per agenti e costruzione dataset

In [14]:
quantiles = {"lvl1": 0.10, "lvl2": 0.25, "lvl5": 0.90}
#prevediamo già k esempi per agente primrio e altri k per agente secondario oracolo (non ancora definito)
primary_examples, oracle_pool = select_examples_by_quantile(
    stats_over_scan,
    quantile_mAP_over_scan,
    quantiles,
    k=1,
    m=2
)
profiles_primary = {
    label: compute_profiles_for_scans(stats_over_scan, primary_examples[label], CATEGORIES)
    for label in primary_examples
}
profiles_oracle = {
    label: compute_profiles_for_scans(stats_over_scan, oracle_pool[label], CATEGORIES)
    for label in oracle_pool
}
#scans da escludere dal dataset di input: quelli già selezionati come esempi primari o come pool di oracle
exclude = set(sum(primary_examples.values(), []) + sum(oracle_pool.values(), []))


In [15]:
dataset_primary = build_input_dataset(
    stats_over_scan,
    exclude,
    SCANS,
    GT_ROOT,
    PRED_CSV,
    SCREENSHOT_DIR
)

In [ ]:
from google.auth import default
creds, project = default()
print("Progetto attivo:", project)


In [9]:
import nest_asyncio
nest_asyncio.apply()

async def test_text():
    agent = LlmAgent(
        name="test_agent",
        description="Test multimodal model with text only",
        model="gemini-2.5-flash",
        instruction="Rispondi brevemente."
    )

    app = App(name="test_app", root_agent=agent)
    runner = InMemoryRunner(app=app)

    await runner.session_service.create_session(
        app_name=app.name,
        user_id="test_user",
        session_id="session_text",
    )

    last = None
    async for event in runner.run_async(
        user_id="test_user",
        session_id="session_text",
        new_message=types.Content(
            role="user",
            parts=[types.Part.from_text(text="Ciao, puoi confermare che sei attivo?")]
        ),
    ):
        last = event

    print(last.content.parts[0].text)

await test_text()


Sì, sono attivo. Sono test_agent.


In [17]:
import mimetypes

def part_from_image_path(path):
    with open(path, "rb") as f:
        data = f.read()
    mime = mimetypes.guess_type(path)[0] or "image/png"
    return types.Part.from_bytes(data=data, mime_type=mime)


In [19]:
async def run_multimodal(runner, session_id, image_path, text):
    msg = types.Content(
        role="user",
        parts=[
            part_from_image_path(image_path),
            types.Part.from_text(text=text)
        ],
    )

    last = None
    async for event in runner.run_async(
        user_id="test_user",
        session_id=session_id,
        new_message=msg,
    ):
        last = event

    return last.content.parts[0].text
from google.adk.apps import App
from google.adk.agents.llm_agent import LlmAgent
from google.adk.runners import InMemoryRunner
from google.genai import types
import nest_asyncio
nest_asyncio.apply()

async def test_multimodal_image_text(image_path):
    agent = LlmAgent(
        name="test_agent_multimodal",
        description="Test multimodal model with image + text",
        model="gemini-2.5-flash",
        instruction="Descrivi brevemente l'immagine."
    )

    app = App(name="test_app_multimodal", root_agent=agent)
    runner = InMemoryRunner(app=app)

    await runner.session_service.create_session(
        app_name=app.name,
        user_id="test_user",
        session_id="session_mm",
    )

    response = await run_multimodal(
        runner,
        session_id="session_mm",
        image_path=image_path,
        text="Puoi descrivere questa immagine?"
    )

    print(response)

#esempio: usa una delle immagini del dataset
await test_multimodal_image_text(dataset_primary[0]["image_pred"])


L'immagine mostra un modello 3D di un'arcata dentale inferiore, vista dall'alto (occlusale), su uno sfondo bianco. I denti sono distintamente colorati per sezioni: i denti posteriori sono blu, quelli intermedi rosa, e quelli anteriori-laterali arancioni, con gli incisivi frontali di colore grigio scuro. Su ogni dente sono visibili numerosi piccoli punti colorati (blu, rosso, verde, giallo, magenta), che probabilmente indicano punti di riferimento anatomici o per misurazioni. Sotto i denti colorati, si intravede una struttura trasparente o grigia chiara che rappresenta la base dell'arcata.
